In [11]:

import cv2
import mediapipe as mp
import time
import scipy
from scipy.stats import bernoulli
import numpy
import random

cap = cv2.VideoCapture(2)

# HANDS PARAMETERS
mpHands = mp.solutions.hands
hands = mpHands.Hands(static_image_mode=False,
                      max_num_hands=2,
                      min_detection_confidence=0.5,
                      min_tracking_confidence=0.5)
mpDraw = mp.solutions.drawing_utils

# FACE PARAMETERS
mp_face_detection = mp.solutions.face_detection
mp_drawing = mp.solutions.drawing_utils
face_detection = mp_face_detection.FaceDetection(
    model_selection=0, min_detection_confidence=0.5)

pTime = 0
cTime = 0

brain = []

rotation_state = 0 #more like angle
rotation_direction = "left"

#def brain_update():



def hands_analysis(res):
    if res.multi_hand_landmarks:#si on a detecte des choses
        if len(res.multi_handedness)==2 :#s'il y a deux mains
            return "left_and_right"
        else :#s'il n'y en a qu'une, laquelle
            if res.multi_handedness[0].classification[0].label=="Left":
                return "right"
            else :
                return "left"
    else :
        return "none"
    
    
    
def face_analysis(res):
    if res.detections:
        
        return "face"
    else:
        return "none"

    
def rotate():
    global rotation_direction
    global rotation_state
    if rotation_direction == "left":
        if rotation_state > -90 :#bord gauche non atteint
            rotation_state -= 1#on continue vers la gauche
            
        else :#changement de direction
            rotation_state += 1
            rotation_direction = "right"
            
    else :
        if rotation_state < 90 :#bord droit non atteint
            rotation_state += 1#on continue vers la droite
            
        else :#changement de direction
            rotation_state -= 1
            rotation_direction = "left"
          
    
    
def random_detection(imgRGB):
    idx = random.randint(0,1)
    if idx == 0:
        return hands_analysis(hands.process(imgRGB))
    if idx == 1:
        return face_analysis(face_detection.process(imgRGB))

def random_forgetting():#une fois sur 300 on enlève un element random au brain
    global brain
    if bernoulli.rvs(1/30, size=1)[0] == 1:
        idx = random.randint(0,len(brain)-1)
        del brain[idx]
        
        
def mini_image_processing(img):
    img=cv2.rotate(img, cv2.ROTATE_90_CLOCKWISE)
    img=cv2.flip(img, 1)
    #img.flags.writeable = False
    imgRGB = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return imgRGB
    
       
person_detected = True

while True:
    
    
    #Boucle de rotation, recherche de visage
    while person_detected == False:
        #print("nobody detected, rotation")
        rotate()
        print([rotation_state,len(brain),"calm"])
        
        random_forgetting()
        
        time.sleep(0.1)#secondes ! On attend avant de rescanner, mais ça dependra de la vitesse de mobilenet
        
        #new frame
        success, img = cap.read()
        imgRGB=mini_image_processing(img)
        
        #nouvelle analyse dans mobilenet pour voir s'il y a une personne
        detection = face_analysis(face_detection.process(imgRGB))
        if detection == face:
            if bernoulli.rvs(1/5, size=1)[0] == 1:
                print("person detected")
                person_detected = True
        
            
    #new frame
    success, img = cap.read()
    imgRGB=mini_image_processing(img)
    
    #detection=random_detection(imgRGB)
    detection = face_analysis(face_detection.process(imgRGB))
    
    
    if detection != "none":#il y a une detection
        
        if detection in brain:#si on l'a deja vue
            print("detection ",detection)
            rotation_state = 0
            print([rotation_state,len(brain),"bored"])
            person_detected = False #on snob
            
        else:#nouvelle detection, on l'ajoute au brain
            print("detection ",detection)
            brain.append(detection)
            rotation_state = 0
            print([rotation_state,len(brain),"excited"])
            time.sleep(3)#on se laisse du temps avant de scanner a nouveau
            
    '''
    results = hands.process(imgRGB)
    #print(results.multi_hand_landmarks)
    if results.multi_hand_landmarks:
        
        for handLms in results.multi_hand_landmarks:
            print(hands_analysis(results.multi_handedness))
            for id, lm in enumerate(handLms.landmark):
                
                h, w, c = img.shape
                cx, cy = int(lm.x *w), int(lm.y*h)
                #if id ==0:
                cv2.circle(img, (cx,cy), 3, (255,0,255), cv2.FILLED)

            mpDraw.draw_landmarks(img, handLms, mpHands.HAND_CONNECTIONS)

    cTime = time.time()
    fps = 1/(cTime-pTime)
    pTime = cTime

    cv2.putText(img,str(int(fps)), (10,70), cv2.FONT_HERSHEY_PLAIN, 3, (255,0,255), 3)
    '''
    #cv2.imshow("Image", img)
    if cv2.waitKey(1) ==ord("q"):
        break
        
        
        
cap.release()
cv2.destroyAllWindows()

[ WARN:0@441.407] global /io/opencv/modules/videoio/src/cap_v4l.cpp (902) open VIDEOIO(V4L2:/dev/video2): can't open camera by index


error: OpenCV(4.6.0) /io/opencv/modules/imgproc/src/color.cpp:182: error: (-215:Assertion failed) !_src.empty() in function 'cvtColor'


In [20]:
import time
x=time.time()
time.sleep(5)
print(time.time()-x)

5.013694763183594
